# STEP 6. 공원 그래프 $G_k = (P, E_k)$ 구축과 구조적 분류

**목적**: STEP 4–5의 관계 자료로 네트워크 정의별·파라미터별 공원 그래프를 만들고, 각 공원을 그래프 구조에 따라 분류한다.

| 정의 | edge 의미 | 파라미터 격자 |
|---|---|---|
| **E0** co-frontage | attachment 세그먼트 공유 또는 폴리곤 접촉 (자명, 별도 레이어) | – |
| **D** walking distance (근접성 기준선) | 보행망 최단거리 $D_{ij} \le d$ (E0 포함) | d ∈ {200, 400, 800} m |
| **N1-abs** angular few-turn | $A_{ij}(R) \le \theta$ (simplest path 길이 ≤ R) | θ ∈ {15, 45, 90, 180}, R ∈ {800, 1600, 3200}, 강건성 {best, ≥2 쌍} |
| **N1-rel** 상대 각도 | 같은 네트워크 거리 구간 안에서 $A_{ij}$가 하위 q | q ∈ {0.1, 0.2} |
| **N2** street-axis continuity | 같은 각도 연속 가로(stroke) 위에 있고, stroke를 따라간 두 공원 구간 사이 간격 ≤ lim (큰길 여부는 STEP 13에서 군집 속성으로) | stroke 각도 ∈ {20, 30, 45}°, lim ∈ {400, 800, 1600} m |
| **N3** excess field overlap | 각도 catchment Jaccard ≥ τ **이고** 메트릭 catchment Jaccard 대비 비율 ≥ κ | θc ∈ {45, 90}, R ∈ {800, 1600}, τ ∈ {0.1, 0.2}, κ ∈ {1.0, 1.25} |

- D를 뺀 모든 정의는 **E0 쌍을 제외**한다(자명한 인접). D는 근접성 자체를 재므로 E0를 포함한다. 같은 선형 공원에서 나온 구간끼리의 쌍도 공원 자신과의 관계이므로 함께 제외한다. N∪E0는 STEP 7 민감도에서 다룬다.
- 대칭화: 두 공원이 모두 출발점을 가지면 $A = \min(A_{ij}, A_{ji})$, 강건 쌍 수는 두 방향 중 큰 값.

**분류(networkx)**: `unresolved` / `isolated` / `dyad` / `chain`(크기 ≥ 3 연결요소지만 2-core 아님) / `core`(2-core 소속) / `periphery`(2-core가 있는 연결요소 안에서 2-core 밖), 그리고 articulation(브리지 공원), degree, core number.

**출력**: `Data/derived/{AREA}/graphs/edges_{def}.parquet`(모든 설정의 edge 목록), `classification.parquet`

In [1]:
# ===== 공통 설정 (모든 STEP 노트북에서 동일) =====
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# 분석 지역: "Brooklyn"(파일럿) 또는 "NYC"(전체). 환경변수 PARK_STUDY_AREA로도 지정 가능.
STUDY_AREA = os.environ.get("PARK_STUDY_AREA", "Brooklyn")

ROOT = Path.cwd()
DATA = ROOT / "Data"
PARKS_PATH = DATA / "NYCPark" / "Parks_Properties_20260923.geojson"
NET_DIR = DATA / "derived" / "network"        # STEP1 결과 (지역 무관, NYC+버퍼 전체)
AREA_DIR = DATA / "derived" / STUDY_AREA      # STEP2 이후 결과
OUT_DIR = ROOT / "outputs" / STUDY_AREA
for _d in (NET_DIR, AREA_DIR, OUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

CRS = 32618            # UTM 18N (m)
R_MAX = 3200           # 탐색 한계 (m) = 보로 네트워크 버퍼 폭
SPEED = 1.0            # speed_m_s=1 -> agg_seconds == 경로 길이(m)

BOROUGHS = {"M": "Manhattan", "X": "Bronx", "B": "Brooklyn", "Q": "Queens", "R": "Staten Island"}
AREA_BOROUGHS = ["B"] if STUDY_AREA == "Brooklyn" else list(BOROUGHS)

# 네트워크 변형: C = 가로 중심선 + 보행전용 경로 (주 분석), S = 보도 포함 전체 보행망 (민감도)
VARIANTS = ["C", "S"]
PRIMARY = "C"
SIDEWALK_FOOTWAYS = {"sidewalk", "crossing", "traffic_island"}

# 공원 필터 (사용자 결정): Flagship + 비공원성 필지 제외, 면적 상한 100 acres (계산은 200 상위집합)
EXCLUDE_TYPES = ["Flagship Park", "Operations", "Lot", "Undeveloped",
                 "Buildings/Institutions", "Cemetery", "Managed Sites"]
AREA_CAP = 100
AREA_CAP_SUPERSET = 200
AREA_CAP_GRID = [50, 100, 200]
RESTRICTED_TYPES = ["Garden", "Jointly Operated Playground"]            # 접근 제한 민감도
STREET_EMBEDDED_TYPES = ["Triangle/Plaza", "Strip", "Mall", "Parkway"]  # 가로 매몰형 민감도

# 공원-네트워크 attachment
ATTACH_BUFFER = {"C": 20.0, "S": 10.0}   # 경계 버퍼 (m): C는 중심선이 도로 폭 절반만큼 떨어져 있음
ATTACH_BUFFER_GRID = {"C": [15.0, 20.0, 30.0], "S": [5.0, 10.0, 20.0]}
INSIDE_SHARE = 0.9       # 공원 내부에 90% 이상 들어간 세그먼트는 attachment에서 제외 (통행은 허용)
K_SOURCES = 8            # 공원당 출발 attachment 최대 개수 (farthest-point sampling)
SMALL_COMPONENT = 200    # 이보다 작은 연결요소에만 붙은 공원은 'unresolved'

# 최종 묶음 공통 절차: 설정별 Louvain(γ) → 보로 단위 퍼콜레이션 필터 → 합의 ≥ 0.5 → 도보 10분 지름 분할
MAIN_GAMMA = 2
COMMERCIAL_BUFFER = 100   # 연구 개요: 군집 구성 공원 주변 100 m = 상업 영역

# 묶음 크기 제약: 묶음 안 모든 공원 쌍이 서로 도보 10분 이내 (보행망 최단거리, 1.33 m/s × 600 s ≈ 800 m)
WALK_LIMIT = 800
WALK_LIMIT_GRID = [400, 800, 1200]   # 5 / 10 / 15분 민감도

# N1 angular few-turn
N1_THETAS = [15, 45, 90, 180]
N1_RADII = [800, 1600, 3200]
N1_REL_Q = [0.1, 0.2]
DIST_BANDS = [0, 200, 400, 800, 1600, 3200]

# D walking distance (근접성 기준선): 보행망 최단거리 ≤ d 이면 연결
D_DISTS = [200, 400, 800]

# N2 street-axis continuity: 같은 연속 가로(stroke) 위, 두 공원 구간 사이 간격 ≤ lim
N2_ALONG = [400, 800, 1600]
# (큰길 속성 계산용) 각도 choice/NACH 반경과 상위 비율
N2_RADII = [800, 1600, 3200]
N2_TOP_P = [0.02, 0.05, 0.10]
STROKE_ANGLE = 30
STROKE_ANGLE_GRID = [20, 30, 45]
N2_MAX_ALONG = [3200, None]

# N3 excess few-turn field overlap
N3_THETAS = [45, 90]
N3_RADII = [800, 1600]
N3_TAU = [0.1, 0.2]
N3_KAPPA = [1.0, 1.25]

print(f"STUDY_AREA={STUDY_AREA}  boroughs={AREA_BOROUGHS}  AREA_DIR={AREA_DIR.relative_to(ROOT)}")

STUDY_AREA=NYC  boroughs=['M', 'X', 'B', 'Q', 'R']  AREA_DIR=Data/derived/NYC


In [2]:
import pickle
import itertools
import networkx as nx
from scipy import sparse

GR_DIR = AREA_DIR / "graphs"
GR_DIR.mkdir(exist_ok=True)
parks = gpd.read_parquet(NET_DIR / "parks.parquet")
e0 = pd.read_parquet(NET_DIR / "pairs_E0.parquet")

# 그래프 노드 = 분석 지역의 공원 (면적 상한 200 상위집합; 상한별 부분집합은 STEP 7에서)
NODES = parks.index[parks.borough.isin(AREA_BOROUGHS)].tolist()
node_set = set(NODES)
print("graph nodes (superset):", len(NODES))


def pair_key(df, a="i", b="j"):
    lo = np.where(df[a] < df[b], df[a], df[b])
    hi = np.where(df[a] < df[b], df[b], df[a])
    return pd.MultiIndex.from_arrays([lo, hi], names=["i", "j"])


def same_parent_pairs(parks):
    """같은 원래 공원(선형 공원 구간)끼리의 쌍: 공원 자신과의 관계이므로 E0처럼 제외."""
    pcs = parks[parks.get("is_piece", False) == True] if "is_piece" in parks else parks.iloc[:0]
    out = set()
    for _, ids in pcs.groupby("parent_id").groups.items():
        ids = sorted(ids)
        out |= {(a, b) for k, a in enumerate(ids) for b in ids[k + 1:]}
    return out


SAME_PARENT = same_parent_pairs(parks)
E0_KEYS = {v: set(pair_key(e0[(e0[f"shared_{v}"] > 0) | e0.touch])) | SAME_PARENT for v in VARIANTS}
print("same-parent piece pairs excluded:", len(SAME_PARENT))
SETTINGS = []   # 엣지가 0개인 설정도 분류하기 위한 목록


def edge_frame(idx, setting):
    SETTINGS.append(setting)
    return pd.DataFrame({"i": idx.get_level_values(0), "j": idx.get_level_values(1), "setting": setting})

graph nodes (superset): 2171
same-parent piece pairs excluded: 1517


## 6-1. N1: angular few-turn (절대 기준 + 상대 기준)

In [3]:
def symmetric_pairs(v, cfg):
    pp = pd.read_parquet(AREA_DIR / f"pairs_{v}.parquet")
    pp = pp[(pp.cfg == cfg) & pp.i.isin(node_set) & pp.j.isin(node_set)].copy()
    pp.index = pair_key(pp)
    num = [c for c in pp.columns if c.startswith(("A_", "D_net", "len_"))]
    cnt = [c for c in pp.columns if c.startswith("n_ok_")]
    g = pp.groupby(level=[0, 1])
    out = g[num].min().join(g[cnt].max())
    out["n_dir"] = g.size()
    return out


N1_ROWS, N1_PAIRS = [], {}
for v in VARIANTS:
    for cfg in pd.read_parquet(AREA_DIR / f"pairs_{v}.parquet", columns=["cfg"]).cfg.unique():
        sp = symmetric_pairs(v, cfg)
        sp = sp[~sp.index.isin(E0_KEYS[v])]
        N1_PAIRS[(v, cfg)] = sp
        for r in N1_RADII:
            for th in N1_THETAS:
                for robust in (False, True):
                    col = f"n_ok_{th}_{r}"
                    m = (sp[f"A_{r}"] <= th) & ((sp[col] >= 2) if robust else True)
                    e = sp.index[m]
                    N1_ROWS.append(edge_frame(e, f"N1abs|{v}|{cfg}|R{r}|th{th}|{'rob' if robust else 'best'}"))
        # 상대 기준: 거리 구간 내 하위 q (R=3200 탐색 결과 사용)
        ok = sp["A_3200"].notna() & sp["D_net"].notna()
        band = pd.cut(sp.loc[ok, "D_net"], DIST_BANDS)
        for q in N1_REL_Q:
            thr = sp.loc[ok, "A_3200"].groupby(band, observed=True).transform(lambda x: x.quantile(q))
            e = sp.loc[ok].index[sp.loc[ok, "A_3200"] <= thr]
            N1_ROWS.append(edge_frame(e, f"N1rel|{v}|{cfg}|q{q}"))
edges_N1 = pd.concat(N1_ROWS, ignore_index=True)
edges_N1.to_parquet(GR_DIR / "edges_N1.parquet")
edges_N1.groupby("setting").size().describe()

count      104.000000
mean      6065.144231
std       7605.551809
min        433.000000
25%       1408.250000
50%       2443.000000
75%       7430.500000
max      32618.000000
dtype: float64

## 6-2. N2: street-axis continuity
연구 개요의 "street-axis continuity"에 맞춰, **같은 연속 가로(stroke) 위**에 있는 공원을 잇는다. 큰길(고 NACH) 여부는 엣지 조건이 아니라 STEP 13의 군집 속성으로 쓴다.
- stroke: STEP 5의 각도 연속 가로. 교차점마다 방향 편차가 각도 한계 미만이면 같은 가로로 이어 붙인다.
- 간격: stroke 위에서 두 공원 attachment 구간 사이의 거리. 구간이 겹치면 0이다.

In [4]:
N2_ROWS = []
for v in [v for v in VARIANTS if (AREA_DIR / f"park_stroke_{v}.parquet").exists()]:
    ps = pd.read_parquet(AREA_DIR / f"park_stroke_{v}.parquet")
    ps = ps[ps.park_id.isin(node_set)]
    for ang in STROKE_ANGLE_GRID:
        m = ps[ps.angle == ang]
        pairs = m.merge(m, on="stroke")
        pairs = pairs[pairs.park_id_x < pairs.park_id_y]
        gap = np.maximum(0, np.maximum(pairs.pos_min_x, pairs.pos_min_y) - np.minimum(pairs.pos_max_x, pairs.pos_max_y))
        pairs = pairs.assign(gap=gap.fillna(0)).groupby(["park_id_x", "park_id_y"])["gap"].min().reset_index()
        pairs.index = pair_key(pairs, "park_id_x", "park_id_y")
        pairs = pairs[~pairs.index.isin(E0_KEYS[v])]
        for lim in N2_ALONG:
            N2_ROWS.append(edge_frame(pairs.index[pairs.gap <= lim], f"N2|{v}|a{ang}|lim{lim}"))
edges_N2 = pd.concat(N2_ROWS, ignore_index=True)
edges_N2.to_parquet(GR_DIR / "edges_N2.parquet")
edges_N2.groupby("setting").size()

setting
N2|C|a20|lim1600    3548
N2|C|a20|lim400     1373
N2|C|a20|lim800     2373
N2|C|a30|lim1600    3774
N2|C|a30|lim400     1413
N2|C|a30|lim800     2480
N2|C|a45|lim1600    3958
N2|C|a45|lim400     1439
N2|C|a45|lim800     2557
dtype: int64

## 6-2b. D: walking distance (근접성 기준선)
연구 개요 RQ3의 "walking distance" 정의: 두 공원 사이 보행망 최단거리(attachment 사이 최솟값)가 d 이하면 연결한다. 근접성 자체를 재므로 E0(맞닿음, 같은 가로 세그먼트)를 포함하고, 같은 공원의 구간 쌍만 뺀다.

In [5]:
D_ROWS = []
for v in VARIANTS:
    cfg = f"d{int(ATTACH_BUFFER[v])}_K{K_SOURCES}"
    sp = symmetric_pairs(v, cfg)
    sp = sp[~sp.index.isin(SAME_PARENT) & sp["D_net"].notna()]
    for dd in D_DISTS:
        D_ROWS.append(edge_frame(sp.index[sp["D_net"] <= dd], f"D|{v}|{cfg}|d{dd}"))
edges_D = pd.concat(D_ROWS, ignore_index=True)
edges_D.to_parquet(GR_DIR / "edges_D.parquet")
edges_D.groupby("setting").size()

setting
D|C|d20_K8|d200     1689
D|C|d20_K8|d400     4142
D|C|d20_K8|d800    11560
D|S|d10_K8|d200     1516
D|S|d10_K8|d400     3929
D|S|d10_K8|d800    11131
dtype: int64

## 6-3. N3: excess few-turn field overlap
공원 × 세그먼트 희소행렬로 길이 가중 교집합을 한 번에 계산한다: $|X_i \cap X_j|_L = (X W X^\top)_{ij}$

$J^{ang}_{ij}$ = 각도 catchment의 길이 가중 Jaccard, $J^{met}_{ij}$ = 같은 R의 메트릭 catchment Jaccard. 비율 $J^{ang}/J^{met}$은 "거리로 기대되는 것보다 두 공원의 few-turn 장이 더 겹치는가"를 뜻한다.

In [6]:
def jaccard_matrix(catch, key, pids, seg_len):
    rows, cols = [], []
    for k, pid in enumerate(pids):
        c = catch[pid].get(key)
        if c is not None and len(c):
            rows.append(np.full(len(c), k)); cols.append(c)
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    X = sparse.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(len(pids), len(seg_len)))
    W = sparse.diags(seg_len)
    inter = (X @ W @ X.T).tocoo()
    size = np.asarray((X @ W).sum(axis=1)).ravel()
    m = inter.row < inter.col
    i, j, iv = inter.row[m], inter.col[m], inter.data[m]
    union = size[i] + size[j] - iv
    return pd.Series(iv / union, index=pd.MultiIndex.from_arrays([np.array(pids)[i], np.array(pids)[j]]))


N3_ROWS, N3_STATS = [], []
for v in VARIANTS:
    seg_len = pd.read_parquet(NET_DIR / f"segments_{v}.parquet", columns=["length"])["length"].values.astype(np.float32)
    with open(AREA_DIR / f"catch_park_{v}.pkl", "rb") as fh:
        catch = pickle.load(fh)
    pids = sorted(set(catch) & node_set)
    jm = {r: jaccard_matrix(catch, ("met", r), pids, seg_len) for r in N3_RADII}
    for th in N3_THETAS:
        for r in N3_RADII:
            ja = jaccard_matrix(catch, ("ang", th, r), pids, seg_len)
            df = pd.DataFrame({"J_ang": ja}).join(jm[r].rename("J_met"), how="left").fillna({"J_met": 0})
            df.index = pair_key(df.reset_index().rename(columns={"level_0": "i", "level_1": "j"}))
            df = df[~df.index.isin(E0_KEYS[v])]
            df["ratio"] = df.J_ang / df.J_met.replace(0, np.nan)
            N3_STATS.append(df.assign(v=v, th=th, r=r))
            for tau in N3_TAU:
                for kappa in N3_KAPPA:
                    e = df.index[(df.J_ang >= tau) & (df.ratio >= kappa)]
                    N3_ROWS.append(edge_frame(e, f"N3|{v}|th{th}|R{r}|tau{tau}|k{kappa}"))
edges_N3 = pd.concat(N3_ROWS, ignore_index=True)
edges_N3.to_parquet(GR_DIR / "edges_N3.parquet")
pd.concat(N3_STATS).reset_index().to_parquet(GR_DIR / "n3_overlap.parquet")
edges_N3.groupby("setting").size().describe()

count     32.000000
mean     114.437500
std      102.024645
min        5.000000
25%       42.000000
50%       78.500000
75%      151.250000
max      405.000000
dtype: float64

## 6-4. 그래프 분류 함수
**연결요소 vs 커뮤니티**: 관대한 설정(θ ≥ 90°, 즉 한 번 꺾기 허용)에서는 브루클린 격자 때문에 공원 대부분이 하나의 거대 연결요소로 **퍼콜레이션**된다. 그 자체가 결과(격자 도시에서 '한 번 꺾어 닿기'는 거의 보편적)지만, 연결요소만으로는 묶음을 구분할 수 없다. 그래서 두 층위로 나눈다.
- **연결 상태**(고립 / 연결요소 / 2-core): 연결요소 기반
- **묶음(group)**: 각 설정 그래프의 **Louvain 커뮤니티**(modularity, seed=0). 내부 연결이 촘촘한 공원 집합을 찾으며, 거대 연결요소 안에서도 사슬 효과 없이 하위 묶음을 분리한다.

역할:
- **isolated**: degree 0 (unresolved는 따로 표시)
- **dyad**: 크기 2 연결요소
- **chain**: 크기 ≥ 3이지만 2-core가 없는 연결요소(나무 구조, A–B–C–D)
- **core**: 2-core 소속 → 내부적으로 순환 연결된 강건한 묶음
- **periphery**: 2-core를 가진 연결요소 안에서 2-core 밖에 매달린 공원
- **articulation**: 제거하면 연결요소가 쪼개지는 공원(브리지 역할)

In [7]:
def classify_graph(nodes, edges_df, unresolved):
    G = nx.Graph()
    G.add_nodes_from(nodes)
    G.add_edges_from(zip(edges_df.i.values, edges_df.j.values))
    core = nx.core_number(G)
    art = set(nx.articulation_points(G))
    rows = {}
    for cid, comp in enumerate(sorted(nx.connected_components(G), key=len, reverse=True)):
        size = len(comp)
        has_core = any(core[n] >= 2 for n in comp)
        for n in comp:
            if size == 1:
                role = "isolated"
            elif size == 2:
                role = "dyad"
            elif core[n] >= 2:
                role = "core"
            elif has_core:
                role = "periphery"
            else:
                role = "chain"
            rows[n] = (cid if size > 1 else -1, size, G.degree(n), core[n], n in art, role)
    out = pd.DataFrame.from_dict(rows, orient="index",
                                 columns=["comp", "comp_size", "degree", "core_number", "articulation", "role"])
    out.loc[out.index.isin(unresolved) & (out.degree == 0), "role"] = "unresolved"
    out["community"], out["community_size"] = -1, 1
    if G.number_of_edges():
        comms = nx.community.louvain_communities(G, seed=0)
        for k, cm in enumerate(sorted(comms, key=len, reverse=True)):
            if len(cm) > 1:
                out.loc[list(cm), "community"] = k
                out.loc[list(cm), "community_size"] = len(cm)
    return out

## 6-5. 모든 설정에 분류 적용
설정 문자열 = `정의|변형|...`. 공원 노드 집합은 면적 상한 200 상위집합이다. 상한 100(주 분석)·50은 노드를 제외하고 다시 분류해야 하므로 여기서 함께 계산한다(`cap` 열).

In [8]:
ALL_EDGES = pd.concat([edges_D, edges_N1, edges_N2, edges_N3], ignore_index=True)
ALL_EDGES["definition"] = ALL_EDGES.setting.str.split("|").str[0]
ALL_EDGES["variant"] = ALL_EDGES.setting.str.split("|").str[1]
unres = {v: set(parks.index[parks[f"unresolved_{v}"]]) for v in VARIANTS}
EDGE_GROUPS = dict(tuple(ALL_EDGES.groupby("setting")))
EMPTY = ALL_EDGES.iloc[:0]
pd.Series(SETTINGS).to_csv(GR_DIR / "settings.csv", index=False, header=["setting"])
cls_rows = []
t0 = time.time()
for cap in AREA_CAP_GRID:
    nodes_cap = [p for p in NODES if parks.at[p, "acres"] <= cap]
    ncap = set(nodes_cap)
    for setting in SETTINGS:
        e = EDGE_GROUPS.get(setting, EMPTY)
        e = e[e.i.isin(ncap) & e.j.isin(ncap)]
        v = setting.split("|")[1]
        c = classify_graph(nodes_cap, e, unres[v])
        c["setting"], c["cap"] = setting, cap
        cls_rows.append(c)
cls = pd.concat(cls_rows).rename_axis("park_id").reset_index()
cls["definition"] = cls.setting.str.split("|").str[0]
cls["variant"] = cls.setting.str.split("|").str[1]
cls.to_parquet(GR_DIR / "classification.parquet")
print(f"{cls.setting.nunique():,} settings × caps classified in {time.time() - t0:.0f}s")

151 settings × caps classified in 37s


## 6-6. 요약: 정의별 역할 분포 (주 변형 C, 상한 100, 대표 설정)

In [9]:
REP = {
    "D": f"D|C|d{int(ATTACH_BUFFER['C'])}_K{K_SOURCES}|d400",
    "N1abs": f"N1abs|C|d{int(ATTACH_BUFFER['C'])}_K{K_SOURCES}|R1600|th45|best",
    "N1rel": f"N1rel|C|d{int(ATTACH_BUFFER['C'])}_K{K_SOURCES}|q0.1",
    "N2": f"N2|C|a{STROKE_ANGLE}|lim800",
    "N3": "N3|C|th90|R800|tau0.2|k1.0",
}
rep = cls[(cls.cap == AREA_CAP) & cls.setting.isin(REP.values())]
tab = rep.pivot_table(index="setting", columns="role", values="park_id", aggfunc="count", fill_value=0)
tab["edges"] = [len(ALL_EDGES[(ALL_EDGES.setting == s)]) for s in tab.index]
tab["components≥3"] = [rep[(rep.setting == s) & (rep.comp_size >= 3)].comp.nunique() for s in tab.index]
tab["largest component"] = [rep[rep.setting == s].comp_size.max() for s in tab.index]
tab["communities≥3"] = [rep[(rep.setting == s) & (rep.community_size >= 3)].community.nunique() for s in tab.index]
tab["largest community"] = [rep[rep.setting == s].community_size.max() for s in tab.index]
tab

role,chain,core,dyad,isolated,periphery,unresolved,edges,components≥3,largest component,communities≥3,largest community
setting,,,,,,,,,,,
D|C|d20_K8|d400,125,1263,170,344,138,17,4142,100,463,118,76
N1abs|C|d20_K8|R1600|th45|best,117,1054,140,510,217,19,2920,72,401,88,86
N1rel|C|d20_K8|q0.1,50,1318,50,434,186,19,9928,29,717,39,200
N2|C|a30|lim800,160,934,164,514,266,19,2480,82,351,100,87
N3|C|th90|R800|tau0.2|k1.0,46,17,92,1883,0,19,106,16,9,16,9


## 6-7. 퍼콜레이션 진단: θ와 R에 따른 최대 연결요소 비율 (N1abs, 주 변형)

In [10]:
n1 = cls[(cls.cap == AREA_CAP) & (cls.definition == "N1abs") & (cls.variant == PRIMARY)
         & cls.setting.str.contains(f"\\|d{int(ATTACH_BUFFER[PRIMARY])}_K{K_SOURCES}\\|") & cls.setting.str.endswith("best")]
perc = n1.groupby("setting").agg(largest=("comp_size", "max"), n=("park_id", "size"),
                                 isolated=("role", lambda r: (r == "isolated").mean()))
perc["largest_share"] = perc.largest / perc.n
perc["R"] = perc.index.str.extract(r"\|R(\d+)\|", expand=False).astype(int).values
perc["theta"] = perc.index.str.extract(r"\|th(\d+)\|", expand=False).astype(int).values
perc.pivot(index="theta", columns="R", values="largest_share").round(2)

R,800,1600,3200
theta,,,
15,0.06,0.07,0.07
45,0.09,0.19,0.21
90,0.20,0.33,0.38
180,0.36,0.86,0.88
